# Milestone 4: What Did It Learn, and What Would It Recommend to You?
Project 2: A Movie Recommendation System — MATH 344L

In this notebook I look inside Model 2 from Milestone 3: I name the hidden factors the SVD found, and I add myself as a new user to see what it recommends.

## 1. Set up

In [1]:
USER, REPO = "JayCBak", "recommendation_system"
!rm -rf {REPO}
!git clone -q https://github.com/{USER}/{REPO}.git

import sys
sys.path.insert(0, f"{REPO}/src")
from evaluate import to_matrix
from models import baseline_matrix, rank_k

This is the same first cell as Milestone 3. It clones my repository and imports `to_matrix` (turns the ratings table into a user-by-movie matrix), `baseline_matrix` (μ + b_user + b_item for every entry), and `rank_k` (best rank-k approximation from an SVD) from my `src` folder.

In [2]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv("ml-100k/u.data", sep="\t",
                      names=["user", "item", "rating", "time"])

titles = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[1])[1].to_numpy()
print(len(titles), titles[49])

1682 Star Wars (1977)


This downloads MovieLens 100K (only if it's not already there) and loads the ratings like in Milestone 3. It also loads the movie titles from `u.item`, where the pieces of each line are separated by `|` and column 1 is the title. `encoding="latin-1"` is needed because some titles have accented letters. Movie number 50 is `titles[49]` (Star Wars) because Python counts from 0.

## 2. Fit Model 2 on all the ratings

In [3]:
R = to_matrix(ratings)
known = ~ np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)

n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies:", popular.sum())

popular movies: 603


This is Model 2 from Milestone 3 (subtract the baseline, put 0 in the gaps, take the SVD), but now it uses **all** the ratings instead of only `train`. In Milestone 3 I was measuring the model, so I had to hide a test set. Now I'm using the model, so it gets all the data. `popular` marks the 603 movies with at least 50 ratings. I only look at those, because a movie with only a few ratings can end up anywhere by chance.

## 3. What did it learn? Naming the hidden factors

In [4]:
def show_factor(f, n=6):
    """Popular movies at the two ends of hidden factor f (f = 1, 2, ...)."""
    v = Vt[f - 1]
    idx = np.where(popular)[0]
    order = idx[np.argsort(v[idx])]
    print(f"Factor {f}, one end:")
    for m in order[:n]:
        print(f"  {v[m]:+.3f}  {titles[m]}")
    print(f"Factor {f}, other end:")
    for m in order[::-1][:n]:
        print(f"  {v[m]:+.3f}  {titles[m]}")

show_factor(1)

Factor 1, one end:
  -0.087  Clockwork Orange, A (1971)
  -0.079  Trainspotting (1996)
  -0.072  Leaving Las Vegas (1995)
  -0.072  Pulp Fiction (1994)
  -0.069  Brazil (1985)
  -0.067  Fargo (1996)
Factor 1, other end:
  +0.211  Independence Day (ID4) (1996)
  +0.175  Return of the Jedi (1983)
  +0.149  Raiders of the Lost Ark (1981)
  +0.149  Back to the Future (1985)
  +0.136  Braveheart (1995)
  +0.134  Forrest Gump (1994)


Each row of `Vt` gives every movie a score on one hidden factor. `show_factor` takes the popular movies, sorts them by their score on factor f with `np.argsort`, and prints the 6 lowest (`order[:n]`) and the 6 highest (`order[::-1][:n]`, the reversed list).

**Factor 1:** one end is dark, strange, R-rated films (A Clockwork Orange, Trainspotting, Leaving Las Vegas, Pulp Fiction). The other end is big, feel-good crowd-pleasers (Independence Day, Return of the Jedi, Raiders of the Lost Ark, Back to the Future, Forrest Gump). Nobody told the SVD about genres or age ratings. It found this split only from who liked what. (The signs could be flipped in a different run, because the SVD can replace u and v by −u and −v without changing anything. The movies at each end would be the same.)

In [5]:
show_factor(3)

Factor 3, one end:
  -0.137  Independence Day (ID4) (1996)
  -0.131  Star Trek: First Contact (1996)
  -0.130  Rock, The (1996)
  -0.129  Alien (1979)
  -0.121  Monty Python and the Holy Grail (1974)
  -0.119  Aliens (1986)
Factor 3, other end:
  +0.163  Sound of Music, The (1965)
  +0.144  Grease (1978)
  +0.099  E.T. the Extra-Terrestrial (1982)
  +0.087  Mary Poppins (1964)
  +0.085  Jaws 2 (1978)
  +0.084  Snow White and the Seven Dwarfs (1937)


**Factor 3:** sci-fi and action at one end (Independence Day, Star Trek: First Contact, The Rock, Alien, Aliens) and musicals and family films at the other (The Sound of Music, Grease, E.T., Mary Poppins, Snow White). Monty Python and Jaws 2 don't really fit their ends, so the factor isn't perfectly clean, but the main idea is clearly "sci-fi/action vs. musicals/family".

## 4. What would it recommend to you?

In [6]:
def find(word):
    """Print the ID and title of every movie whose title contains word."""
    for m, t in enumerate(titles):
        if word.lower() in t.lower():
            print(m + 1, t)

find("alien")

176 Aliens (1986)
183 Alien (1979)
343 Alien: Resurrection (1997)
665 Alien 3 (1992)


`find` searches the titles for a word so I can look up movie IDs. `enumerate` gives each title with its position, `.lower()` makes the search ignore capital letters, and it prints `m + 1` because movie IDs start at 1.

In [7]:
my_ratings = {50: 5, 172: 5, 183: 5, 195: 5, 89: 4, 1: 4,
              143: 1, 451: 1, 88: 1, 739: 2}

for movie, stars in my_ratings.items():
    print(stars, "stars:", titles[movie - 1])

5 stars: Star Wars (1977)
5 stars: Empire Strikes Back, The (1980)
5 stars: Alien (1979)
5 stars: Terminator, The (1984)
4 stars: Blade Runner (1982)
4 stars: Toy Story (1995)
1 stars: Sound of Music, The (1965)
1 stars: Grease (1978)
1 stars: Sleepless in Seattle (1993)
2 stars: Pretty Woman (1990)


This is a dictionary that pairs each movie ID with my rating (1–5 stars). I rated 10 older movies: I like sci-fi (Star Wars, The Empire Strikes Back, Alien, The Terminator, Blade Runner) and Toy Story, and I don't like romance and musicals (The Sound of Music, Grease, Sleepless in Seattle, Pretty Woman). The loop prints the titles so I can check I used the right IDs.

In [8]:
me = np.full((1, R.shape[1]), np.nan)
for movie, stars in my_ratings.items():
    me[0, movie - 1] = stars
R_me = np.vstack([R, me])
print(R_me.shape)

known_me = ~ np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)
U, s, Vt = np.linalg.svd(X_me, full_matrices=False)
P = rank_k(U, s, Vt, 10) + B_me

(944, 1682)


`me` is a one-row matrix that is empty except for my 10 ratings. `np.vstack` adds it under R as user 944, so the matrix is now 944 × 1682. Then it's Model 2 again with k = 10 (the k the handout uses; my best in Milestone 3 was k = 15, with k = 10 almost as good). My predictions are the last row, `P[-1]`.

In [9]:
mine = P[-1]
base = B_me[-1]
can_pick = popular & np.isnan(R_me[-1])

def top(scores, n=10):
    """The n highest-scoring movies among those we may recommend."""
    return np.argsort(-np.where(can_pick, scores, -np.inf))[:n]

print("Model 2 recommends:")
for m in top(mine):
    print(f"  {mine[m]:.2f}  {titles[m]}")
print("Baseline recommends:")
for m in top(base):
    print(f"  {base[m]:.2f}  {titles[m]}")
print("in both lists:", len(set(top(mine)) & set(top(base))))

Model 2 recommends:
  4.28  Close Shave, A (1995)
  4.26  Wrong Trousers, The (1993)
  4.20  Fargo (1996)
  4.15  Casablanca (1942)
  4.14  Raiders of the Lost Ark (1981)
  4.11  Wallace & Gromit: The Best of Aardman Animation (1996)
  4.08  Godfather, The (1972)
  4.07  Schindler's List (1993)
  4.07  Shawshank Redemption, The (1994)
  4.06  Return of the Jedi (1983)
Baseline recommends:
  4.23  Close Shave, A (1995)
  4.15  Wrong Trousers, The (1993)
  4.13  Schindler's List (1993)
  4.13  Shawshank Redemption, The (1994)
  4.09  Casablanca (1942)
  4.08  Wallace & Gromit: The Best of Aardman Animation (1996)
  4.08  Usual Suspects, The (1995)
  4.06  Good Will Hunting (1997)
  4.01  Titanic (1997)
  4.01  12 Angry Men (1957)
in both lists: 6


`mine` is my row of Model 2 predictions and `base` is my row of baseline predictions. `can_pick` is True only for popular movies I haven't rated, since there's no point recommending a movie I've already seen. `top` gives every other movie a score of −∞ so it can't be picked, and `np.argsort(-...)` sorts from highest to lowest. `set(...) & set(...)` keeps the movies that are in both lists.

**6 of the 10 movies are the same in both lists.** Both lists are mostly movies that almost everyone rates highly (A Close Shave, The Wrong Trousers, Casablanca, Schindler's List). The baseline already knows those, and the SVD only adjusts them. Model 2 swapped in Fargo, Raiders of the Lost Ark, The Godfather and Return of the Jedi for The Usual Suspects, Good Will Hunting, Titanic and 12 Angry Men.

In [10]:
personal = mine - base
print("Moved up most for me:")
for m in top(personal, 6):
    print(f"  {personal[m]:+.2f}  {titles[m]}")
print("Moved down most for me:")
for m in top(-personal, 6):
    print(f"  {personal[m]:+.2f}  {titles[m]}")

Moved up most for me:
  +0.34  Fargo (1996)
  +0.34  Return of the Jedi (1983)
  +0.25  Star Trek: The Wrath of Khan (1982)
  +0.22  Star Trek: First Contact (1996)
  +0.21  2001: A Space Odyssey (1968)
  +0.20  Raiders of the Lost Ark (1981)
Moved down most for me:
  -0.36  English Patient, The (1996)
  -0.21  Liar Liar (1997)
  -0.19  Dirty Dancing (1987)
  -0.19  Mr. Holland's Opus (1995)
  -0.18  Ghost (1990)
  -0.17  Mrs. Doubtfire (1993)


`mine - base` is exactly the part the SVD added for me (the personal part). `top(personal, 6)` gives the 6 movies it moved up the most, and `top(-personal, 6)` gives the 6 it moved down the most.

From only 10 ratings, the model pushed sci-fi up (Return of the Jedi, two Star Trek movies, 2001: A Space Odyssey) and romance/drama down (The English Patient, Dirty Dancing, Ghost), without being told what genre anything is. The changes are small: at most about a third of a star.

**Key idea:** a prediction = baseline + personal part. The baseline (how good the movie is and how generous I am) is the big part. The SVD's personal part is a correction of a few tenths of a star. That's why Model 2 only beat the baseline by about 3% in Milestone 3, and also why its recommendations make sense.

## 5. My finding

**Finding:** Without being told any genres, the SVD discovered a "sci-fi/action vs. musicals/family" factor. From just 10 of my ratings it moved Return of the Jedi and Star Trek up by about 0.3 stars and The English Patient down by 0.36 stars, but 6 of its top 10 picks were the same crowd favorites that the simple average recommends.

# Exercises

### Exercise 1: Name the factors

In [11]:
# refit on R (without me), as in Section 2
U, s, Vt = np.linalg.svd(X, full_matrices=False)
show_factor(2)

Factor 2, one end:
  -0.099  Independence Day (ID4) (1996)
  -0.092  Twister (1996)
  -0.077  Eraser (1996)
  -0.074  Air Force One (1997)
  -0.067  Saint, The (1997)
  -0.066  Phenomenon (1996)
Factor 2, other end:
  +0.203  Star Wars (1977)
  +0.199  Pulp Fiction (1994)
  +0.173  Godfather, The (1972)
  +0.168  Empire Strikes Back, The (1980)
  +0.151  Fargo (1996)
  +0.146  Return of the Jedi (1983)


Factor 2: one end is 1996–97 summer blockbusters (Independence Day, Twister, Eraser, Air Force One, The Saint). The other end is famous, highly praised films (Star Wars, Pulp Fiction, The Godfather, The Empire Strikes Back, Fargo).

My names for the factors:
- **Factor 1: "edgy cult films vs. feel-good crowd-pleasers"**
- **Factor 2: "acclaimed classics vs. new popcorn blockbusters"**
- **Factor 3: "sci-fi/action vs. musicals/family"**

**Factor 2 was the hardest to name**, because its "classics" end mixes very different genres (space opera, crime, mafia drama), and Star Wars and Independence Day are both sci-fi but sit at opposite ends. It seems to be more about quality or reputation than about genre.

Some factors might not match any idea a person would have because the SVD doesn't look for ideas. It finds the directions that explain the most variation in the ratings, and each one has to be orthogonal to the ones before it. A real taste can be split across several factors, or one factor can mix several tastes together, like "older, acclaimed" and "not a 1996 blockbuster". The factors are also affected by who happens to be in the data (mostly 1990s users) and by the 94% of the matrix that is just zeros.

### Exercise 2: Your own ratings

With my ratings (sci-fi fan, dislikes romance and musicals):

- **Overlap:** 6 of my top 10 Model 2 picks are also in the baseline's top 10.
- **Do the lists make sense?** Mostly yes. The "moved up" list is mostly sci-fi and adventure (Return of the Jedi, Star Trek II and First Contact, 2001, Raiders of the Lost Ark), which fits my 5-star ratings for Star Wars, Empire, Alien and The Terminator. The "moved down" list is romance and sentimental dramas (The English Patient, Dirty Dancing, Ghost, Mr. Holland's Opus), which fits my 1-star ratings for Grease, The Sound of Music and Sleepless in Seattle.
- **One recommendation I think is wrong: Fargo** (moved up the most, +0.34, and #3 in my Model 2 list). It's a dark crime comedy, not sci-fi, and nothing I rated is like it. I think the model made it because of factor 2: Fargo sits at the same "acclaimed classics" end as Star Wars and The Empire Strikes Back. Many people who loved Star Wars also loved Fargo, so my 5-star Star Wars ratings pushed Fargo up too. The model learns which movies are liked by the same people, not why they are liked. Liar Liar being moved down is a similar case: it's a comedy, not a romance, but it seems to be liked by the same people who liked Grease and Sleepless in Seattle.

### Exercise 3: The opposite person

In [12]:
def personal_lists(ratings_dict, k=10, n=6):
    """Add one user, refit Model 2, and return their lists."""
    row = np.full((1, R.shape[1]), np.nan)
    for movie, stars in ratings_dict.items():
        row[0, movie - 1] = stars
    R_new = np.vstack([R, row])
    known_new = ~ np.isnan(R_new)
    B_new = baseline_matrix(R_new)
    X_new = np.where(known_new, R_new - B_new, 0.0)
    U, s, Vt = np.linalg.svd(X_new, full_matrices=False)
    pred = (rank_k(U, s, Vt, k) + B_new)[-1]
    base_new = B_new[-1]
    pers = pred - base_new
    return {"up": [titles[m] for m in top(pers, n)],
            "down": [titles[m] for m in top(-pers, n)],
            "base top": [titles[m] for m in top(base_new)],
            "base scores": base_new}

swap = {5: 1, 1: 5}
friend_ratings = {movie: swap.get(stars, stars) for movie, stars in my_ratings.items()}
print("friend:", friend_ratings)

me_lists = personal_lists(my_ratings)
friend_lists = personal_lists(friend_ratings)

print("\nFriend - moved up:  ", friend_lists["up"])
print("Friend - moved down:", friend_lists["down"])
print("\nMy 'up' movies now in friend's 'down':", sorted(set(me_lists["up"]) & set(friend_lists["down"])))
print("My 'down' movies now in friend's 'up':", sorted(set(me_lists["down"]) & set(friend_lists["up"])))

print("\nBaseline top 10 the same?", me_lists["base top"] == friend_lists["base top"])
shift = friend_lists["base scores"] - me_lists["base scores"]
print("baseline scores shift by:", round(shift[can_pick].min(), 4), "to", round(shift[can_pick].max(), 4))

friend: {50: 1, 172: 1, 183: 1, 195: 1, 89: 4, 1: 4, 143: 5, 451: 5, 88: 5, 739: 2}



Friend - moved up:   ['English Patient, The (1996)', "Mr. Holland's Opus (1995)", 'Dirty Dancing (1987)', 'Dead Man Walking (1995)', 'First Wives Club, The (1996)', 'Ghost (1990)']
Friend - moved down: ['Return of the Jedi (1983)', 'Raiders of the Lost Ark (1981)', 'Star Trek: The Wrath of Khan (1982)', 'Star Trek: First Contact (1996)', 'Aliens (1986)', 'Fargo (1996)']

My 'up' movies now in friend's 'down': ['Fargo (1996)', 'Raiders of the Lost Ark (1981)', 'Return of the Jedi (1983)', 'Star Trek: First Contact (1996)', 'Star Trek: The Wrath of Khan (1982)']
My 'down' movies now in friend's 'up': ['Dirty Dancing (1987)', 'English Patient, The (1996)', 'Ghost (1990)', "Mr. Holland's Opus (1995)"]

Baseline top 10 the same? True
baseline scores shift by: -0.4 to -0.4


The friend has the same 10 movies, but every 5 became a 1 and every 1 became a 5 (the 4s and the 2 stay the same).

**The lists swap almost completely.** Fargo, Raiders of the Lost Ark, Return of the Jedi and both Star Trek movies go from my "moved up" list to the friend's "moved down" list. The English Patient, Dirty Dancing, Ghost and Mr. Holland's Opus go from my "moved down" list to the friend's "moved up" list. This makes sense because the personal part is linear: flipping the friend's ratings around the middle flips the sign of most of the residuals in their row, so the projection onto the factors mostly flips sign too.

**The baseline top 10 does not change.** It's exactly the same 10 movies in the same order. The only thing about the friend that the baseline knows is how generous they are (b_user). The friend's average is lower (2.9 vs. 3.3), so every baseline prediction for them shifts down by the same 0.40 stars. Adding the same number to every movie doesn't change the order. The baseline has no idea what kind of movies a person likes, and that's exactly the part the SVD adds.

### Exercise 4: Does k matter here?

In [13]:
for k in [2, 10, 50]:
    lists = personal_lists(my_ratings, k=k)
    print(f"k = {k}")
    print("  moved up:  ", lists["up"])
    print("  moved down:", lists["down"])

k = 2
  moved up:   ['Pulp Fiction (1994)', 'Godfather, The (1972)', 'Fargo (1996)', '2001: A Space Odyssey (1968)', 'Raising Arizona (1987)', 'Leaving Las Vegas (1995)']
  moved down: ['Independence Day (ID4) (1996)', 'Twister (1996)', 'Air Force One (1997)', 'Phenomenon (1996)', 'Eraser (1996)', 'Saint, The (1997)']


k = 10
  moved up:   ['Fargo (1996)', 'Return of the Jedi (1983)', 'Star Trek: The Wrath of Khan (1982)', 'Star Trek: First Contact (1996)', '2001: A Space Odyssey (1968)', 'Raiders of the Lost Ark (1981)']
  moved down: ['English Patient, The (1996)', 'Liar Liar (1997)', 'Dirty Dancing (1987)', "Mr. Holland's Opus (1995)", 'Ghost (1990)', 'Mrs. Doubtfire (1993)']


k = 50
  moved up:   ['Return of the Jedi (1983)', 'Aliens (1986)', 'People vs. Larry Flynt, The (1996)', 'Godfather, The (1972)', 'Fantasia (1940)', 'Bridge on the River Kwai, The (1957)']
  moved down: ['When Harry Met Sally... (1989)', 'Mary Poppins (1964)', 'Dirty Dancing (1987)', 'Dead Poets Society (1989)', 'My Fair Lady (1964)', 'While You Were Sleeping (1995)']


In [14]:
for k in [2, 10, 50]:
    row = np.full((1, R.shape[1]), np.nan)
    for movie, stars in my_ratings.items():
        row[0, movie - 1] = stars
    R_new = np.vstack([R, row])
    B_new = baseline_matrix(R_new)
    X_new = np.where(~np.isnan(R_new), R_new - B_new, 0.0)
    U, s, Vt = np.linalg.svd(X_new, full_matrices=False)
    pers = rank_k(U, s, Vt, k)[-1]
    print(f"k = {k:2d}  largest personal change: {np.abs(pers[can_pick]).max():.2f} stars"
          f"   typical size: {np.abs(pers[can_pick]).mean():.3f} stars")

k =  2  largest personal change: 0.15 stars   typical size: 0.019 stars


k = 10  largest personal change: 0.36 stars   typical size: 0.047 stars


k = 50  largest personal change: 0.43 stars   typical size: 0.063 stars


- **k = 2:** the "moved up" list becomes Pulp Fiction, The Godfather, Fargo, 2001, Raising Arizona, Leaving Las Vegas, and the "moved down" list becomes the 1996–97 blockbusters (Independence Day, Twister, Air Force One). With only 2 factors, the model can only describe me with factors 1 and 2 ("cult/acclaimed vs. blockbusters"), so it misses that I'm a sci-fi fan who dislikes romance. The changes are also tiny (at most 0.15 stars). This is like the small k in Milestone 3: **too simple, so it misses real patterns** (underfitting).
- **k = 10:** sci-fi moves up and romance moves down, which is the list that makes the most sense for my ratings.
- **k = 50:** the lists get mixed and harder to explain: The People vs. Larry Flynt, Fantasia and The Bridge on the River Kwai move up, and Dead Poets Society, My Fair Lady and When Harry Met Sally move down, next to some sci-fi and musicals that still make sense. The changes also get bigger (up to 0.43 stars). With 50 factors but only 10 ratings from me, the model has enough freedom to fit small, accidental patterns in my 10 ratings. This is the **overfitting** I saw in Milestone 3, where the test RMSE went back up for large k.

So k matters here the same way it did for RMSE: a medium k like 10 gives the most sensible recommendations.